In [1]:
import os
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd

In [2]:
import os
import requests
from bs4 import BeautifulSoup
import csv

## Detect ./data folder
if not os.path.exists('./data'):
    os.makedirs('./data')

urls = ["https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-new-local",
        "https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-new-non-local",
        "https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-continuing-local",
        "https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-continuing-non-local"]

def crawl_tbl_border(url, filename):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    ## Only catch class = "tbl-border"
    table = soup.find('table', class_='tbl-border')

    rows = table.find_all('tr')
    data = []
    for row in rows:
        cols = row.find_all(['td', 'th'])
        cols = [ele.text.strip() for ele in cols]
        data.append(cols)

    # If value is missing, fill it with "N/A"
    for i in range(len(data)):
        for j in range(len(data[i])):
            if data[i][j] == '':
                data[i][j] = 'N/A'
            if data[i][j] == '—':
                data[i][j] = 'N/A'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerows(data)

    return data

## Detect ./data/charges folder
path = './data/hall_info/charges'
if not os.path.exists(path):
    os.makedirs(path)
    
for url in urls:
    if "new-local" in url:
        crawl_tbl_border(url, path + "/local_new.csv")
    elif "new-non-local" in url:
        crawl_tbl_border(url, path + "/non_local_new.csv")
    elif "continuing-local" in url:
        crawl_tbl_border(url, path + "/local_continuing.csv")
    elif "continuing-non-local" in url:
        crawl_tbl_border(url, path + "/non_local_continuing.csv")

In [3]:
urls = ["https://shrl.hkust.edu.hk/apply-for-housing/ug/number-of-hall-places-and-room-types"]
# Detect ./data/room_types folder
path = "./data/hall_info/room_types"
if not os.path.exists(path):
    os.makedirs(path)
    
for url in urls:
    crawl_tbl_border(url, path + "/room_types.csv")

In [4]:
import pandas as pd

def crawl_hall_info_member(url, filename):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    ## Only catch table with class = profile-image
    table = soup.find('table', class_='profile-image')
    rows = table.find_all('tr')
    prof_data = [["Name","Position","Tel","Email"]]
    for row in rows:
        cols = row.find_all(['td', 'th'])
        cols = [ele.text.strip() for ele in cols]

        # delete "Tel:" and "Email:"
        cols = [ele.replace("Tel: ", "").replace("Email: ", "") for ele in cols]
        # delete empty cols
        cols = [ele for ele in cols if ele]
        # seperate the cols with \n
        cols = [ele.split('\n') for ele in cols][0]
        prof_data.append(cols)
    ## Only catch all table with class = gray-line-table team 
    
    tables = soup.find_all('table', class_='gray-line-table team')
    team_data = []
    for table in tables:
        data = []
        caption = table.find('caption')
        position = caption.text.strip() if caption else 'N/A'
        thead = table.find('thead')
        headers = [th.text.strip() for th in thead.find_all('th')]
        rows = table.find_all('tr')
        data.append(["Position"] + headers)
        for row in rows:
            cols = row.find_all('td')
            cols = [ele.text.strip() for ele in cols]
            if cols:
                data.append([position] + cols)
        
        team_data.append(data)
    


 
    df = pd.DataFrame(prof_data[1:], columns=prof_data[0])
    for t in team_data:
        df_t = pd.DataFrame(t[1:], columns=t[0])
        df = pd.concat([df, df_t], ignore_index=True)
    
    # fill missing value with "N/A"
    df = df.fillna("N/A")
    df = df.replace("", "N/A")

    df.to_csv(os.path.join(filename, url.split('/')[-1] + '_member' + '.csv'), index=False)
    return df
    

urls = ["https://shrl.hkust.edu.hk/residential-halls/ug/ughall1",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall2",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall3",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall4",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall5",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall6",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall7",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall8",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall9",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall-jch"]

# Detect ./data/hall_info folder
if not os.path.exists('./data/hall_info/member'):
    os.makedirs('./data/hall_info/member')
    
for url in urls:
    crawl_hall_info_member(url, "./data/hall_info/member")



In [5]:
def crawl_facilities(url, filename):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # find started item number i, with class_='field__item mtpc-section-item mtpc-section-item--{i}'
    # Where its child has class = "accordion-question"
    for i in range(1,20):
        item = soup.find('div', class_=f'field__item mtpc-section-item mtpc-section-item--{i}')
        if item:
            question = item.find('div', class_='accordion-question')
            if question:
                start_item_num = i + 1
                break
    print(start_item_num)
    facilities = []
    columns = []
    item = soup.find('div', class_=f'field__item mtpc-section-item mtpc-section-item--{start_item_num}')
    if item:
        ## Catch <ul> under item
        ul = item.find('ul')
        if ul:
            lis = ul.find_all('li')
            for li in lis:
                columns.append(li.text.strip())

    start_item_num += 1
    facilities_num = len(columns)


    ## Only catch class = "field__item mtpc-section-item mtpc-section-item--i" where i = 8,9,10,11,12,13
    for i in range(start_item_num , start_item_num + facilities_num):
        item = soup.find('div', class_=f'field__item mtpc-section-item mtpc-section-item--{i}')
        if item:
            text = item.text.strip()
            # Remove all "Text Area " in text
            text = text.replace("Text Area", "")
            text = text.replace("\n", " ").strip()
            facilities.append(text)

    # Error dectection
    try:
        facilities = pd.DataFrame([facilities], columns=columns)
    except ValueError as e:
        print(f"Error processing {url}: {e}")
        return facilities,columns
    facilities.to_csv(os.path.join(filename, url.split('/')[-1] + '_facilities' + '.csv'), index=False)
    
    return facilities

urls = ["https://shrl.hkust.edu.hk/residential-halls/ug/ughall1",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall2",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall3",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall4",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall5",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall6",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall7",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall8",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall9",
        "https://shrl.hkust.edu.hk/residential-halls/ug/ughall-jch"]


# Detect ./data/hall_info/facilities folder
if not os.path.exists('./data/hall_info/facilities'):
    os.makedirs('./data/hall_info/facilities')
for url in urls:
    crawl_facilities(url, "./data/hall_info/facilities/")


8
8
8
8
8
8
8
8
8
9


In [6]:
# Crawl key dates
def crawl_key_dates(url, filename):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    ## Only catch class = mtpc-2col-item mtpc-2col-item--2

    items = soup.find_all('div', class_='mtpc-2col-item mtpc-2col-item--2')

    data = []
    # Catch table with class = calendar-details
    for item in items:
        table = item.find('table', class_='calendar-details')
        if table:
            rows = table.find_all('tr')
            for row in rows:
                cols = row.find_all(['td', 'th'])
                cols = [ele.text.strip() for ele in cols]
                # Replace /xa with space
                cols = [ele.replace('\xa0', ' ') for ele in cols]
                data.append(cols)

    data = pd.DataFrame(data, columns=["Date", "Event"])
    data.to_csv(filename, index=False)

    
    return data

# Detect ./data/hall_info/announcement folder
path = './data/announcement'
if not os.path.exists(path):
    os.makedirs(path)

crawl_key_dates("https://shrl.hkust.edu.hk/apply-for-housing/ug/new-local-ugs",path + "/key_dates.csv")      

,Date,Event
0,4 Sep 2025,Payment Deadline for Fall Installment (RY2025-...
1,8 Oct 2025,Hall Extension Result Announcement - Long Dist...
2,26 Nov 2025,Payment Deadline for Spring Installment (RY202...
3,15 Dec 2025,Hall Offer Announcement (RY2025-26 Spring-only...
4,16 Dec - 18 Dec 2025,Hall Offer Acceptance Period (RY2025-26 Spring...
5,9 Jan 2026,Payment Deadline for Spring Installment (RY202...
6,10 Jan 2026,Check-out (RY2025-26 Fall-only Offer)
7,26 Jan 2026,Check-in (RY2025-26 Spring-only Offer)
8,2 Jun 2026,Check-out (RY2025-26)
9,5 - 12 Aug 2026,Hall Application Period (RY2026-27)


In [7]:
base_url = "https://shrl.hkust.edu.hk"
urls = ["https://shrl.hkust.edu.hk/admission-policy/ug/priority-housing",
       "https://shrl.hkust.edu.hk/admission-policy/ug/hall-point-system-i",
       "https://shrl.hkust.edu.hk/admission-policy/ug/lottery",
       "https://shrl.hkust.edu.hk/admission-policy/ug/waitlist"]

def crawl_admission_methods(url, filename):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    ## Only catch class = mtpc-2col-item mtpc-2col-item--1
    data = []

    items = soup.find_all('div', class_='mtpc-2col-item mtpc-2col-item--1')
    for item in items:
        paragraphs = item.find_all(['p','table'])
        for p in paragraphs:
            if p.name == 'p':
                text = p.text.strip()
                # Replace /xa with space
                text = text.replace('\xa0', ' ')
                text = text.replace('\n', ' ').strip()
                data.append([text])
            if p.name == 'table':
                
                rows = p.find_all('tr')
                for row in rows:
                    cols = row.find_all(['td', 'th'])
                    cols = [ele.text.strip() for ele in cols]
                    # Replace /xa with space
                    cols = [ele.replace('\xa0', ' ') for ele in cols]
                    # If cols is not empty, append to data
                    if cols:
                        data.append(cols)
        # find <a class = "text-btn"> with href, if it can download file, download it
        a_tags = item.find_all('a', class_='text-btn', href=True)
        for a in a_tags:
            file_url = a['href']
            if file_url.endswith('.pdf') or file_url.endswith('.docx') or file_url.endswith('.doc'):
                file_name = file_url.split('/')[-1]
                file_path = os.path.join(os.path.dirname(filename), file_name)
                # Download the file
                file_url = base_url + file_url
                print(file_url)
                file_response = requests.get(file_url)
                with open(file_path, 'wb') as f:
                    for d in file_response.iter_content():
                        f.write(d)
    # Save to csv     
    df = pd.DataFrame(data)

    # Delete empty rows
    df = df[df.apply(lambda x: x.str.strip().astype(bool).any(), axis=1)]
    
    df.to_csv(filename, index=False, header=False)
    return df

path = "./data/policy/"
if not os.path.exists(path):
    os.makedirs(path)

for url in urls:
    title = url.split('/')[-1]
    data = crawl_admission_methods(url, path + title +".csv")


https://shrl.hkust.edu.hk/upload/UG_Hall_Policy/Annex_1-Guideline_for_Special_Application_with_Exceptional_Hardship.pdf
https://shrl.hkust.edu.hk/upload/UG_Hall_Policy/Annex_2-Guideline_for_Part_1-Priority_Housing.pdf
https://shrl.hkust.edu.hk/upload/UG_Hall_Policy/Example-Part_1.Priority_Housing.pdf
https://shrl.hkust.edu.hk/upload/Waiting_List/20260129_CUG_2526_Local.pdf
https://shrl.hkust.edu.hk/upload/Waiting_List/20260129_CUG_2526_NL.NHB_.pdf


In [2]:
# Convert outputs to Markdown and consolidate by category (crawler-only)
import os
from pathlib import Path
import json
import pandas as pd
import zipfile


def df_to_markdown(df: pd.DataFrame) -> str:
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        vals = [str(v).replace("\n", " ").replace("|", "\\|") for v in row.values]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def write_text(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def convert_csv_to_md(src: Path, dst_dir: Path) -> Path:
    try:
        df = pd.read_csv(src, dtype=str, on_bad_lines="skip")
    except Exception as e:
        df = pd.DataFrame([[f"Failed to read CSV: {e}"]], columns=["Error"])
    df = df.fillna("")
    md = f"# {src.name}\n\n" + df_to_markdown(df)
    dst = dst_dir / (src.stem + ".md")
    write_text(dst, md)
    return dst


def convert_json_to_md(src: Path, dst_dir: Path) -> Path:
    try:
        with open(src, "r", encoding="utf-8") as f:
            data = json.load(f)
        pretty = json.dumps(data, ensure_ascii=False, indent=2)
        md = f"# {src.name}\n\n```json\n{pretty}\n```\n"
    except Exception as e:
        md = f"# {src.name}\n\nError loading JSON: {e}\n"
    dst = dst_dir / (src.stem + ".md")
    write_text(dst, md)
    return dst


def convert_binary_stub_to_md(src: Path, dst_dir: Path) -> Path:
    size = src.stat().st_size if src.exists() else 0
    md = (
        f"# {src.name}\n\n"
        f"- Original path: {src}\n"
        f"- Size: {size} bytes\n"
        f"- Note: Binary/document file; embedded as reference.\n"
    )
    dst = dst_dir / (src.stem + ".md")
    write_text(dst, md)
    return dst


def consolidate_to_markdown(target_root: str = "./outputs_md_crawler"):
    root = Path(target_root)
    # Only include folders generated by this crawler notebook
    categories = {
        Path("./data/hall_info/charges"): "charges",
        Path("./data/hall_info/room_types"): "room_types",
        Path("./data/hall_info/member"): "member",
        Path("./data/hall_info/facilities"): "facilities",
        Path("./data/announcement"): "announcement",
        Path("./data/policy"): "policy",
    }
    allowed_exts = {".csv", ".json", ".faiss", ".pdf", ".doc", ".docx"}

    processed = []
    for src_dir, cat in categories.items():
        dst_dir = root / cat
        if not src_dir.exists():
            continue
        for src in src_dir.rglob("*"):
            if not src.is_file():
                continue
            if src.suffix.lower() not in allowed_exts:
                continue
            try:
                if src.suffix.lower() == ".csv":
                    dst = convert_csv_to_md(src, dst_dir)
                elif src.suffix.lower() == ".json":
                    dst = convert_json_to_md(src, dst_dir)
                else:
                    dst = convert_binary_stub_to_md(src, dst_dir)
                processed.append((src, dst))
            except Exception as e:
                err_md = dst_dir / (src.stem + "_ERROR.md")
                write_text(err_md, f"# {src.name}\n\nConversion failed: {e}\n")
                processed.append((src, err_md))

    print(f"✅ Converted {len(processed)} files to Markdown under {root.resolve()}")
    for src, dst in processed[:10]:
        print(f" - {src} -> {dst}")
    return processed


def zip_folder(folder_path: str, zip_path: str | None = None) -> Path:
    folder = Path(folder_path)
    if zip_path is None:
        zip_path = str(folder.with_suffix(".zip"))
    zip_p = Path(zip_path)
    with zipfile.ZipFile(zip_p, "w", zipfile.ZIP_DEFLATED) as z:
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(folder))
    print(f"📦 Zipped to {zip_p.resolve()}")
    return zip_p

# Run single-step: convert & consolidate to Markdown, then zip (crawler-only)
_ = consolidate_to_markdown("./outputs_md_crawler")
_ = zip_folder("./outputs_md_crawler")

✅ Converted 37 files to Markdown under /Users/wangjunyuan/桌面副本/fyp /outputs_md_crawler
 - data/hall_info/charges/local_new.csv -> outputs_md_crawler/charges/local_new.md
 - data/hall_info/charges/local_continuing.csv -> outputs_md_crawler/charges/local_continuing.md
 - data/hall_info/charges/non_local_continuing.csv -> outputs_md_crawler/charges/non_local_continuing.md
 - data/hall_info/charges/non_local_new.csv -> outputs_md_crawler/charges/non_local_new.md
 - data/hall_info/room_types/room_types.csv -> outputs_md_crawler/room_types/room_types.md
 - data/hall_info/member/ughall8_member.csv -> outputs_md_crawler/member/ughall8_member.md
 - data/hall_info/member/ughall5_member.csv -> outputs_md_crawler/member/ughall5_member.md
 - data/hall_info/member/ughall-jch_member.csv -> outputs_md_crawler/member/ughall-jch_member.md
 - data/hall_info/member/ughall2_member.csv -> outputs_md_crawler/member/ughall2_member.md
 - data/hall_info/member/ughall7_member.csv -> outputs_md_crawler/member/ugh